# 02 — Preprocesamiento
Unión de tablas CRM, limpieza, feature engineering, encoding y split train/test.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from paths import PROCESSED_DIR, load_crm_tables, merge_crm_tables

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
accounts, products, sales_teams, pipeline = load_crm_tables()

## Limpieza — filtrar filas del CSV crudo

En el CSV (`sales_pipeline.csv`) vemos filas como estas:

| Tipo en el CSV | Ejemplo | ¿Qué hacemos? |
|----------------|---------|---------------|
| **Won** con fechas y valor | `Cancity, Won, 2017-03-01, 1054` | **Se conserva** |
| **Lost** con `close_value = 0` | `Gekko & Co, Lost, ..., 0` | **Se conserva** |
| **Engaging** sin `close_date` | cuenta vacía o fechas incompletas | **Se elimina** — deal aún abierto |
| **Prospecting** con columnas vacías | `Anna Snelling, Prospecting,,,` | **Se elimina** — no es Won ni Lost |

**Código:** nos quedamos solo con `Won` y `Lost`, y quitamos filas sin `account` o `engage_date`.

También en `paths.py` corregimos typos al cargar: `GTXPro` → `GTX Pro`, `technolgy` → `technology`.

In [2]:
df = merge_crm_tables(accounts, products, sales_teams, pipeline)

# Solo oportunidades cerradas (Won / Lost)
df = df[df["deal_stage"].isin(["Won", "Lost"])].copy()
df = df.dropna(subset=["account", "engage_date"])

df["engage_date"] = pd.to_datetime(df["engage_date"])
df["won"] = (df["deal_stage"] == "Won").astype(int)

print(f"Registros tras limpieza: {len(df)}")
print(f"Nulos en sales_price tras merge: {df['sales_price'].isnull().sum()}")

Registros tras limpieza: 6711
Nulos en sales_price tras merge: 0


## Limpieza — completar valores faltantes y crear variables

Después de unir las tablas (pipeline + accounts + products + sales_teams), algunos campos numéricos pueden venir vacíos. **No eliminamos esas filas**; rellenamos con la **mediana** del sector (o mediana global).

Ejemplos de lo que creamos aquí:
- `company_age` = año del deal − año de fundación de la empresa
- `revenue_per_employee` = ingresos ÷ empleados (evitando dividir por 0)
- `engage_month`, `engage_quarter` = partes de la fecha para capturar estacionalidad

**Nota:** `close_value` **no** se usa como variable del modelo (evita filtrar el resultado: en Lost siempre es 0).

In [3]:
df["subsidiary_of"] = df["subsidiary_of"].fillna("None")
df["has_subsidiary"] = (df["subsidiary_of"] != "None").astype(int)

for col in ["revenue", "employees"]:
    df[col] = df.groupby("sector")[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median())

df["company_age"] = df["engage_date"].dt.year - df["year_established"]
df["revenue_per_employee"] = df["revenue"] / df["employees"].replace(0, np.nan)
df["revenue_per_employee"] = df["revenue_per_employee"].fillna(df["revenue_per_employee"].median())
df["engage_year"] = df["engage_date"].dt.year
df["engage_month"] = df["engage_date"].dt.month
df["engage_quarter"] = df["engage_date"].dt.quarter
df["engage_dayofweek"] = df["engage_date"].dt.dayofweek
df["is_premium_product"] = (df["sales_price"] > df["sales_price"].median()).astype(int)

## Partición train / test (80 % / 20 %)

Separamos los datos **después** de limpiar:
- **80 %** → entrenamiento (`X_train`, `y_train`)
- **20 %** → prueba (`X_test`, `y_test`) — el modelo **no** los ve al entrenar

`test_size=0.2` y `stratify=y` mantienen la misma proporción Won/Lost en ambos conjuntos.

In [4]:
cat_cols = ["sector", "office_location", "subsidiary_of", "product", "series",
            "manager", "regional_office", "sales_agent"]
num_cols = ["year_established", "revenue", "employees", "sales_price",
            "company_age", "revenue_per_employee", "engage_year", "engage_month",
            "engage_quarter", "engage_dayofweek", "has_subsidiary", "is_premium_product"]

for col in cat_cols:
    df[col] = df[col].astype(str)

X = pd.get_dummies(df[cat_cols + num_cols], columns=cat_cols, drop_first=True)
y = df["won"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Features: {X_train.shape[1]} | Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Features: 86 | Train: 5368 | Test: 1343


## Encoding y normalización (scaling)

Son **dos pasos distintos** después de la limpieza:

### Encoding — texto → números
Columnas como `product`, `sales_agent`, `sector` son texto. Con **One-Hot Encoding** (`get_dummies`) cada categoría pasa a columnas 0/1.

Ejemplo: `product = GTX Basic` → columna `product_GTX Basic = 1`, las demás productos = 0.

### Normalización (scaling) — igualar escalas numéricas
Variables como `revenue` (millones) y `engage_month` (1–12) tienen rangos muy distintos. **StandardScaler** las centra y escala para que ninguna domine solo por ser más grande.

**¿Cuándo lo usamos?**
| Paso | Random Forest (nb. 03) | SVM (nb. 04) |
|------|-------------------------|--------------|
| Encoding | Sí | Sí |
| Scaling | **No** necesita | **Sí** — usa `X_train_scaled` |

El scaler se ajusta **solo** con train (`fit`) y se aplica igual al test (`transform`).

In [5]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
X_train_scaled.to_csv(PROCESSED_DIR / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(PROCESSED_DIR / "X_test_scaled.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)
joblib.dump(scaler, PROCESSED_DIR / "scaler.joblib")

print("Datos procesados guardados en:", PROCESSED_DIR)

Datos procesados guardados en: D:\UNIVALLE\GESTION 1-2026 ISI\TECN.EMERG.I\ASSIGNMENT\2026.06.14 TI26 - Proyecto + Exposición Final\REPOSITORIO-TI26-TEAM-Heredia-Illanes\data\processed
